## **Hyperparameter Optimization with Nested Cross-Validation**

This notebook performs robust model selection for pIC₅₀ prediction using nested cross-validation. The goal is to mitigate overfitting by separating the model evaluation process (outer loop) from the hyperparameter tuning (inner loop).

### Key Steps:
- Load REU-selected feature dataset for a target.
- Define multiple machine learning models with parameter grids:
  - Random Forest
  - XGBoost
  - Gradient Boosting
  - Support Vector Regressor (SVR)
  - K-Nearest Neighbors (KNN)
  - Multi-Layer Perceptron (MLP)
- Perform **nested CV** with 5 outer folds and 3 inner folds.
- Report the average test R² and RMSE from the outer loop.
- Evaluate final model on the external test set and save results.

This pipeline ensures more reliable generalization estimates and reduces bias from tuning on the same data used for evaluation.


In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import GridSearchCV, KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import r2_score, mean_squared_error
from xgboost import XGBRegressor
from math import sqrt
import warnings


In [3]:
warnings.filterwarnings("ignore")

In [4]:
# === Paths to datasets ===

reu_path = "/content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/data/descriptors/reduced_features_reu_2"
meta_cols = ['molecule_chembl_id', 'canonical_smiles', 'activity_class', 'pIC50']

In [5]:
targets = ["5-HT6","ache", "bace1", "buche","esr1","gsk-3beta", "mao-b"]

target=targets[5]

In [6]:
df_train = pd.read_csv(f"{reu_path}/{target}_train_reu.csv").dropna(how="any")
df_test = pd.read_csv(f"{reu_path}/{target}_test_reu.csv").dropna(how="any")

In [7]:
X_train = df_train.drop(columns=meta_cols)
y_train = df_train["pIC50"]
X_test = df_test.drop(columns=meta_cols)
y_test = df_test["pIC50"]

In [8]:
# === Models and Hyperparameters ===
models = {
    "RandomForest": (
        RandomForestRegressor(random_state=42),
        {"model__n_estimators": [100, 200],
         "model__max_depth": [None, 10]}
    ),
    "XGBoost": (
        XGBRegressor(objective="reg:squarederror", random_state=42, verbosity=0),
        {"model__n_estimators": [100, 200],
         "model__max_depth": [3, 5]}
    ),
    "GradientBoosting": (
        GradientBoostingRegressor(random_state=42),
        {"model__n_estimators": [100, 200],
         "model__learning_rate": [0.05, 0.1]}
    ),
    "SVR": (
        SVR(),
        {"model__C": [1, 10],
         "model__kernel": ["rbf"]}
    ),
    "KNeighborsRegressor": (
        KNeighborsRegressor(),
        {"model__n_neighbors": [5, 7]}
    ),
    "MLPRegressor": (
        MLPRegressor(max_iter=1000, random_state=42),
        {"model__hidden_layer_sizes": [(64,), (64, 32)],
         "model__alpha": [0.0001, 0.001]}
    )
}


In [9]:
# === Nested CV Setup ===
outer_cv = KFold(n_splits=5, shuffle=True, random_state=42)
inner_cv = KFold(n_splits=3, shuffle=True, random_state=42)

results = []

for name, (model, param_grid) in models.items():
    print(f"\n Optimizing: {name}")

    # Apply scaling where needed
    if name in ["SVR", "KNeighborsRegressor", "MLPRegressor"]:
        pipeline = Pipeline([
            ("scaler", StandardScaler()),
            ("model", model)
        ])
    else:
        pipeline = Pipeline([
            ("model", model)
        ])

    grid = GridSearchCV(pipeline, param_grid, cv=inner_cv, scoring="r2", n_jobs=-1)

    # Nested CV R²
    nested_scores = cross_val_score(grid, X_train, y_train, cv=outer_cv, scoring="r2", n_jobs=-1)
    mean_nested_r2 = np.mean(nested_scores)

    # Nested CV RMSE (Note: RMSE must be converted from negative)
    nested_rmse_scores = cross_val_score(grid, X_train, y_train, cv=outer_cv, scoring="neg_root_mean_squared_error", n_jobs=-1)
    mean_nested_rmse = -np.mean(nested_rmse_scores)  # Take negative to get actual RMSE

    # === Fit on Full Training Set with Best Hyperparameters ===
    grid.fit(X_train, y_train)
    best_model = grid.best_estimator_

    # === Evaluate on Test Set ===
    y_pred_test = best_model.predict(X_test)
    test_r2 = r2_score(y_test, y_pred_test)
    test_rmse = sqrt(mean_squared_error(y_test, y_pred_test))

    results.append({
        "Target": target,
        "Model": name,
        "Best Params": grid.best_params_,
        "Nested CV R2": round(mean_nested_r2, 4),
        "Nested CV RMSE": round(mean_nested_rmse, 4),
        "Test R2": round(test_r2, 4),
        "Test RMSE": round(test_rmse, 4)
    })




 Optimizing: RandomForest

 Optimizing: XGBoost

 Optimizing: GradientBoosting

 Optimizing: SVR

 Optimizing: KNeighborsRegressor

 Optimizing: MLPRegressor


In [10]:
# === Save and Display ===
df_results = pd.DataFrame(results)


display(df_results)

,Target,Model,Best Params,Nested CV R2,Nested CV RMSE,Test R2,Test RMSE
0,gsk-3beta,RandomForest,"{'model__max_depth': None, 'model__n_estimator...",0.6323,0.7522,0.6551,0.7440
1,gsk-3beta,XGBoost,"{'model__max_depth': 3, 'model__n_estimators':...",0.6179,0.7666,0.6567,0.7423
2,gsk-3beta,GradientBoosting,"{'model__learning_rate': 0.1, 'model__n_estima...",0.6058,0.7789,0.6164,0.7846
3,gsk-3beta,SVR,"{'model__C': 10, 'model__kernel': 'rbf'}",0.6590,0.7241,0.6746,0.7227
4,gsk-3beta,KNeighborsRegressor,{'model__n_neighbors': 5},0.6084,0.7763,0.6364,0.7639
5,gsk-3beta,MLPRegressor,"{'model__alpha': 0.0001, 'model__hidden_layer_...",0.4394,0.9294,0.4989,0.8968


In [11]:
# === Save results ===

results_out = f"/content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/Hyperparameter-optimization-traditional-ML/optimized_hyperparameters_traditional_{target}_reu_nested_CV.csv"
df_results.to_csv(results_out, index=False)

print("\n Saved results to:", results_out)


 Saved results to: /content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/Hyperparameter-optimization-traditional-ML/optimized_hyperparameters_traditional_gsk-3beta_reu_nested_CV.csv
